In [ ]:
"""
Customer Churn Prediction — Streamlit App
Loads the trained sklearn Pipeline (preprocessing + model bundled together)
and predicts churn probability for a single customer entered via the UI.

Run with:  streamlit run app.py
Requires:  models/churn_pipeline.pkl to exist (produced by Section 11 of the notebook)
"""

import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Customer Churn Predictor", page_icon="📉", layout="centered")


@st.cache_resource
def load_model():
    """Cached so the pipeline is loaded from disk only once per session, not on every rerun."""
    return joblib.load("models/churn_pipeline.pkl")


try:
    model = load_model()
except FileNotFoundError:
    st.error(
        "models/churn_pipeline.pkl not found. Run Section 11 of the notebook first "
        "to train and save the pipeline, then place the file in a `models/` folder "
        "next to this app."
    )
    st.stop()

st.title("📉 Customer Churn Predictor")
st.caption("Logistic Regression (class_weight='balanced') — trained on IBM Telco Customer Churn data")

st.markdown("Enter a customer's details below to estimate their churn risk.")

with st.form("customer_form"):
    st.subheader("Account details")
    col1, col2 = st.columns(2)
    with col1:
        tenure = st.number_input("Tenure (months)", min_value=0, max_value=100, value=12)
        monthly_charges = st.number_input("Monthly charges ($)", min_value=0.0, value=70.0, step=0.5)
    with col2:
        total_charges = st.number_input("Total charges ($ to date)", min_value=0.0, value=840.0, step=1.0)
        contract = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])

    st.subheader("Demographics")
    col3, col4, col5 = st.columns(3)
    with col3:
        senior = st.selectbox("Senior citizen", ["No", "Yes"])
    with col4:
        partner = st.selectbox("Has partner", ["No", "Yes"])
    with col5:
        dependents = st.selectbox("Has dependents", ["No", "Yes"])

    st.subheader("Services")
    col6, col7 = st.columns(2)
    with col6:
        phone_service = st.selectbox("Phone service", ["Yes", "No"])
        multiple_lines = st.selectbox("Multiple lines", ["No", "Yes", "No phone service"])
        internet_service = st.selectbox("Internet service", ["DSL", "Fiber optic", "No"])
        online_security = st.selectbox("Online security", ["No", "Yes", "No internet service"])
        online_backup = st.selectbox("Online backup", ["No", "Yes", "No internet service"])
    with col7:
        device_protection = st.selectbox("Device protection", ["No", "Yes", "No internet service"])
        tech_support = st.selectbox("Tech support", ["No", "Yes", "No internet service"])
        streaming_tv = st.selectbox("Streaming TV", ["No", "Yes", "No internet service"])
        streaming_movies = st.selectbox("Streaming movies", ["No", "Yes", "No internet service"])

    st.subheader("Billing")
    col8, col9 = st.columns(2)
    with col8:
        paperless_billing = st.selectbox("Paperless billing", ["Yes", "No"])
    with col9:
        payment_method = st.selectbox(
            "Payment method",
            ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"],
        )

    submitted = st.form_submit_button("Predict churn risk", use_container_width=True)

if submitted:
    # ── Engineer the same features created in Section 5 of the notebook ────────
    # tenure_group: must use the exact same bins/labels the pipeline was trained on
    tenure_group = pd.cut(
        [tenure], bins=[0, 12, 24, 48, 72], labels=["0-12mo", "13-24mo", "25-48mo", "49-72mo"]
    )[0]
    if pd.isna(tenure_group):
        tenure_group = "0-12mo"  # tenure=0 edge case falls outside the first bin's open interval

    charge_per_month = total_charges / (tenure + 1)
    high_risk_fiber = int(internet_service == "Fiber optic" and monthly_charges > 70)

    # ── Build the row with exactly the columns the pipeline expects ────────────
    # (TotalCharges itself is NOT included — it was dropped after engineering charge_per_month)
    row = pd.DataFrame([{
        "SeniorCitizen": 1 if senior == "Yes" else 0,
        "Partner": partner,
        "Dependents": dependents,
        "tenure": tenure,
        "PhoneService": phone_service,
        "MultipleLines": multiple_lines,
        "InternetService": internet_service,
        "OnlineSecurity": online_security,
        "OnlineBackup": online_backup,
        "DeviceProtection": device_protection,
        "TechSupport": tech_support,
        "StreamingTV": streaming_tv,
        "StreamingMovies": streaming_movies,
        "Contract": contract,
        "PaperlessBilling": paperless_billing,
        "PaymentMethod": payment_method,
        "MonthlyCharges": monthly_charges,
        "tenure_group": tenure_group,
        "charge_per_month": charge_per_month,
        "high_risk_fiber": high_risk_fiber,
    }])

    prob = model.predict_proba(row)[0, 1]
    pred = model.predict(row)[0]

